In [1]:
# AI-Powered Data Cleaning & Analysis Studio (Tkinter)
# Single-file advanced project
# Save as: ai_data_studio.py

import os
import re
import threading
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
from tkinter.scrolledtext import ScrolledText

import matplotlib
matplotlib.use("TkAgg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk

try:
    from sklearn.ensemble import IsolationForest
    from sklearn.cluster import KMeans
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import r2_score
except ImportError:
    IsolationForest = None
    KMeans = None
    LinearRegression = None
    r2_score = None

# ===============================
# OPTIONAL AI SUMMARY (Local Rule-Based)
# ===============================
def smart_column_type(series):
    s = series.dropna().astype(str).str.strip()
    if s.empty:
        return "unknown"
    if pd.api.types.is_numeric_dtype(series):
        return "numeric"
    dt = pd.to_datetime(series, errors='coerce', dayfirst=True)
    if dt.notna().mean() > 0.7:
        return "date"
    unique_ratio = s.nunique() / max(len(s), 1)
    avg_len = s.str.len().mean()
    if unique_ratio < 0.2 and avg_len < 30:
        return "categorical"
    return "text"


def detect_possible_id(series):
    s = series.dropna().astype(str)
    if len(s) == 0:
        return False
    unique_ratio = s.nunique() / len(s)
    return unique_ratio > 0.95


class AIDataStudio:
    
    def replace_values(self):
        if self.df is None:
            messagebox.showwarning("Warning", "Load data first")
            return

        # STEP 1: SELECT COLUMN
        select_win = tk.Toplevel(self.root)
        select_win.title("Select Column")
        select_win.geometry("300x150")

        tk.Label(select_win, text="Select Column").pack(pady=10)

        col_var = tk.StringVar()
        combo = ttk.Combobox(select_win, textvariable=col_var,
                            values=list(self.df.columns), state="readonly")
        combo.pack(pady=5)

        def proceed():
            col = col_var.get()
            if not col:
                messagebox.showerror("Error", "Select a column")
                return
            select_win.destroy()
            self.open_replace_window(col)

        ttk.Button(select_win, text="Next", command=proceed).pack(pady=10)
    def open_replace_window(self, column):
        win = tk.Toplevel(self.root)
        win.title(f"Replace Values - {column}")
        win.geometry("500x500")

        unique_vals = self.df[column].dropna().astype(str).unique()

        frame = ttk.Frame(win)
        frame.pack(fill="both", expand=True, padx=10, pady=10)

        canvas = tk.Canvas(frame)
        scrollbar = ttk.Scrollbar(frame, orient="vertical", command=canvas.yview)

        scroll_frame = ttk.Frame(canvas)

        scroll_frame.bind(
            "<Configure>",
            lambda e: canvas.configure(scrollregion=canvas.bbox("all"))
        )

        canvas.create_window((0, 0), window=scroll_frame, anchor="nw")
        canvas.configure(yscrollcommand=scrollbar.set)

        canvas.pack(side="left", fill="both", expand=True)
        scrollbar.pack(side="right", fill="y")

        entries = {}

        # SHOW UNIQUE VALUES
        for val in unique_vals:
            row = ttk.Frame(scroll_frame)
            row.pack(fill="x", pady=3)

            tk.Label(row, text=str(val), width=25, anchor="w").pack(side="left")

            entry = ttk.Entry(row, width=25)
            entry.pack(side="right", padx=5)

            entries[val] = entry

        # ✅ FIXED FUNCTIONS (INSIDE METHOD)
        def apply_changes():
            for old_val, entry in entries.items():
                new_val = entry.get().strip()

                if new_val == "":
                    continue

                self.df[column] = self.df[column].astype(str).replace(old_val, new_val)

            self.root.after(0, lambda: self.show_dataframe(self.df))
            messagebox.showinfo("Success", "Values replaced successfully")

        def on_close():
            if messagebox.askyesno("Exit", "Save current changes and exit?"):
                win.destroy()

        btn_frame = ttk.Frame(win)
        btn_frame.pack(pady=10)

        ttk.Button(btn_frame, text="Apply", command=apply_changes).pack(side="left", padx=5)
        ttk.Button(btn_frame, text="Close", command=on_close).pack(side="left", padx=5)

        win.protocol("WM_DELETE_WINDOW", on_close)
    def __init__(self, root):
        self.root = root
        self.root.title("AI Data Cleaning & Analysis Studio - Advanced")
        self.root.geometry("1600x920")
        self.root.configure(bg="#f4f7fb")

        self.original_df = None
        self.df = None
        self.filtered_df = None
        self.current_file = None
        self.current_chart = None
        self.colorbar = None
        self.sort_states = {}

        self.setup_style()
        self.create_layout()

    # ===============================
    # UI SETUP
    # ===============================
    def setup_style(self):
        style = ttk.Style()
        style.theme_use("clam")
        style.configure("TFrame", background="#f4f7fb")
        style.configure("TLabel", background="#f4f7fb", font=("Segoe UI", 10))
        style.configure("Header.TLabel", font=("Segoe UI", 16, "bold"), foreground="#1f2937")
        style.configure("TButton", font=("Segoe UI", 10, "bold"), padding=6)
        style.configure("TNotebook", background="#f4f7fb")
        style.configure("Treeview", rowheight=26, font=("Segoe UI", 9))
        style.configure("Treeview.Heading", font=("Segoe UI", 10, "bold"))
        style.map("TButton", background=[("active", "#dbeafe")])

    def create_layout(self):
        top = ttk.Frame(self.root, padding=10)
        top.pack(fill="x")

        ttk.Label(top, text="AI-Powered Data Cleaning & Analysis Studio", style="Header.TLabel").pack(side="left", padx=8)

        btn_frame = ttk.Frame(top)
        btn_frame.pack(side="right")

        ttk.Button(btn_frame, text="Load CSV/Excel", command=self.load_file).pack(side="left", padx=5)
        ttk.Button(btn_frame, text="Auto Clean (AI)", command=self.run_thread(self.auto_clean)).pack(side="left", padx=5)
        ttk.Button(btn_frame, text="Replace Values", command=self.replace_values).pack(side="left", padx=5)
        ttk.Button(btn_frame, text="Set Format", command=self.open_set_format_dialog).pack(side="left", padx=5)
        ttk.Button(btn_frame, text="Generate Analysis", command=self.run_thread(self.generate_analysis)).pack(side="left", padx=5)
        ttk.Button(btn_frame, text="Export Clean Data", command=self.export_clean_data).pack(side="left", padx=5)
        ttk.Button(btn_frame, text="Reset", command=self.reset_data).pack(side="left", padx=5)

        search_frame = ttk.Frame(self.root, padding=(10, 0, 10, 8))
        search_frame.pack(fill="x")

        ttk.Label(search_frame, text="Search:").pack(side="left", padx=(0, 5))
        self.search_var = tk.StringVar()
        search_entry = ttk.Entry(search_frame, textvariable=self.search_var, width=30)
        search_entry.pack(side="left", padx=5)
        search_entry.bind("<KeyRelease>", lambda e: self.search_data())

        ttk.Label(search_frame, text="Filter Column:").pack(side="left", padx=(15, 5))
        self.filter_col_var = tk.StringVar()
        self.filter_col_combo = ttk.Combobox(search_frame, textvariable=self.filter_col_var, width=22, state="readonly")
        self.filter_col_combo.pack(side="left", padx=5)

        ttk.Label(search_frame, text="Condition:").pack(side="left", padx=(15, 5))
        self.condition_var = tk.StringVar(value="contains")
        self.condition_combo = ttk.Combobox(search_frame, textvariable=self.condition_var, width=18, state="readonly",
                                            values=["contains", "equals", "starts with", "ends with", ">", "<", ">=", "<="])
        self.condition_combo.pack(side="left", padx=5)

        ttk.Label(search_frame, text="Value:").pack(side="left", padx=(15, 5))
        self.filter_val_var = tk.StringVar()
        ttk.Entry(search_frame, textvariable=self.filter_val_var, width=20).pack(side="left", padx=5)
        ttk.Button(search_frame, text="Apply Filter", command=self.apply_filter).pack(side="left", padx=5)
        ttk.Button(search_frame, text="Clear Filter", command=self.clear_filter).pack(side="left", padx=5)

        ttk.Label(search_frame, text="Sort:").pack(side="left", padx=(15, 5))
        self.sort_col_var = tk.StringVar()
        self.sort_col_combo = ttk.Combobox(search_frame, textvariable=self.sort_col_var, width=22, state="readonly")
        self.sort_col_combo.pack(side="left", padx=5)
        ttk.Button(search_frame, text="Low → High", command=lambda: self.sort_data(True)).pack(side="left", padx=5)
        ttk.Button(search_frame, text="High → Low", command=lambda: self.sort_data(False)).pack(side="left", padx=5)

        self.notebook = ttk.Notebook(self.root)
        self.notebook.pack(fill="both", expand=True, padx=10, pady=5)

        self.tab_data = ttk.Frame(self.notebook)
        self.tab_analysis = ttk.Frame(self.notebook)
        self.tab_visual = ttk.Frame(self.notebook)
        self.tab_ai = ttk.Frame(self.notebook)

        self.notebook.add(self.tab_data, text="Data View")
        self.notebook.add(self.tab_analysis, text="Analysis Report")
        self.notebook.add(self.tab_visual, text="Visualization")
        self.notebook.add(self.tab_ai, text="AI Insights")

        self.create_data_tab()
        self.create_analysis_tab()
        self.create_visual_tab()
        self.create_ai_tab()

        self.status_var = tk.StringVar(value="Ready")
        status = ttk.Label(self.root, textvariable=self.status_var, relief="sunken", anchor="w")
        status.pack(fill="x", side="bottom")

    def create_data_tab(self):
        frame = ttk.Frame(self.tab_data, padding=8)
        frame.pack(fill="both", expand=True)

        tree_frame = ttk.Frame(frame)
        tree_frame.pack(fill="both", expand=True)

        self.tree = ttk.Treeview(tree_frame, show="headings")
        self.tree.pack(side="left", fill="both", expand=True)

        y_scroll = ttk.Scrollbar(tree_frame, orient="vertical", command=self.tree.yview)
        x_scroll = ttk.Scrollbar(frame, orient="horizontal", command=self.tree.xview)
        self.tree.configure(yscrollcommand=y_scroll.set, xscrollcommand=x_scroll.set)
        y_scroll.pack(side="right", fill="y")
        x_scroll.pack(fill="x")

    def create_analysis_tab(self):
        self.analysis_text = ScrolledText(self.tab_analysis, font=("Consolas", 11), wrap="word")
        self.analysis_text.pack(fill="both", expand=True, padx=10, pady=10)

    def create_visual_tab(self):
        main = ttk.Frame(self.tab_visual, padding=8)
        main.pack(fill="both", expand=True)

        controls = ttk.Frame(main)
        controls.pack(fill="x", pady=5)

        ttk.Label(controls, text="Chart Type:").pack(side="left", padx=5)
        self.chart_type = tk.StringVar(value="Histogram")
        self.chart_combo = ttk.Combobox(controls, textvariable=self.chart_type, state="readonly", width=20,
                                        values=["Histogram", "Bar", "Line", "Scatter", "Boxplot", "Pie", "Correlation Heatmap", "Outliers"])
        self.chart_combo.pack(side="left", padx=5)

        ttk.Label(controls, text="X Column:").pack(side="left", padx=5)
        self.x_var = tk.StringVar()
        self.x_combo = ttk.Combobox(controls, textvariable=self.x_var, state="readonly", width=20)
        self.x_combo.pack(side="left", padx=5)

        ttk.Label(controls, text="Y Column:").pack(side="left", padx=5)
        self.y_var = tk.StringVar()
        self.y_combo = ttk.Combobox(controls, textvariable=self.y_var, state="readonly", width=20)
        self.y_combo.pack(side="left", padx=5)

        ttk.Button(controls, text="Plot", command=self.plot_chart).pack(side="left", padx=10)
        ttk.Button(controls, text="Save Chart", command=self.save_chart).pack(side="left", padx=5)

        self.fig = plt.Figure(figsize=(10, 6), dpi=100)
        self.ax = self.fig.add_subplot(111)
        self.canvas = FigureCanvasTkAgg(self.fig, master=main)
        self.canvas.get_tk_widget().pack(fill="both", expand=True)
        toolbar = NavigationToolbar2Tk(self.canvas, main)
        toolbar.update()

    def create_ai_tab(self):
        self.ai_text = ScrolledText(self.tab_ai, font=("Consolas", 11), wrap="word")
        self.ai_text.pack(fill="both", expand=True, padx=10, pady=10)

    # ===============================
    # HELPERS
    # ===============================
    def on_ui_thread(self):
        return threading.current_thread() is threading.main_thread()

    def run_on_ui(self, func, *args, **kwargs):
        if self.on_ui_thread():
            return func(*args, **kwargs)
        self.root.after(0, lambda: func(*args, **kwargs))

    def set_status(self, msg):
        def update_status():
            self.status_var.set(msg)
            self.root.update_idletasks()
        self.run_on_ui(update_status)

    def run_thread(self, func):
        def wrapper():
            threading.Thread(target=func, daemon=True).start()
        return wrapper

    def get_active_df(self):
        return self.filtered_df.copy() if self.filtered_df is not None else (self.df.copy() if self.df is not None else None)

    # ===============================
    # FILE LOAD
    # ===============================
    def load_file(self):
        file_path = filedialog.askopenfilename(filetypes=[("CSV files", "*.csv"), ("Excel files", "*.xlsx *.xls")])
        if not file_path:
            return
        try:
            self.set_status("Loading file...")
            if file_path.endswith(".csv"):
                df = pd.read_csv(file_path, encoding="utf-8", low_memory=False)
            else:
                df = pd.read_excel(file_path)

            self.original_df = df.copy()
            self.df = df.copy()
            self.filtered_df = None
            self.current_file = file_path
            self.update_combos()
            self.show_dataframe(self.df)
            self.analysis_text.delete("1.0", tk.END)
            self.ai_text.delete("1.0", tk.END)
            self.set_status(f"Loaded: {os.path.basename(file_path)} | Rows: {len(self.df)} | Cols: {len(self.df.columns)}")
            messagebox.showinfo("Success", "File loaded successfully.")
        except Exception as e:
            messagebox.showerror("Error", f"Could not load file:\n{e}")
            self.set_status("Load failed")

    def update_combos(self):
        if self.df is None:
            return
        cols = list(self.df.columns)
        for combo in [self.filter_col_combo, self.sort_col_combo, self.x_combo, self.y_combo]:
            combo["values"] = cols
        if cols:
            self.filter_col_var.set(cols[0])
            self.sort_col_var.set(cols[0])
            self.x_var.set(cols[0])
            self.y_var.set(cols[min(1, len(cols)-1)])

    # ===============================
    # DISPLAY
    # ===============================
    def show_dataframe(self, df):
        if not self.on_ui_thread():
            self.root.after(0, lambda data=df.copy(): self.show_dataframe(data))
            return

        self.tree.delete(*self.tree.get_children())
        self.tree["columns"] = list(df.columns)

        for col in df.columns:
            self.tree.heading(col, text=col)
            self.tree.column(col, width=140, anchor="center")

        sample_df = df.head(500)
        for _, row in sample_df.iterrows():
            self.tree.insert("", "end", values=list(row.astype(str)))

    # ===============================
    # AUTO CLEANING ENGINE
    # ===============================
    def auto_clean(self):
        if self.df is None:
            self.run_on_ui(messagebox.showwarning, "Warning", "Please load a file first.")
            return

        self.set_status("AI cleaning in progress...")
        df = self.df.copy()
        log = []

        # 1. Standardize column names
        old_cols = df.columns.tolist()
        df.columns = [re.sub(r'[^A-Za-z0-9_]+', '_', str(c).strip()).strip('_').title() for c in df.columns]
        log.append("1) Standardized column names")

        # 2. Remove full duplicate rows
        before = len(df)
        df = df.drop_duplicates()
        log.append(f"2) Removed duplicate rows: {before - len(df)}")

        # 3. Trim strings and clean blanks
        for col in df.columns:
            if df[col].dtype == object:
                df[col] = df[col].astype(str).str.strip()
                df[col] = df[col].replace({"": np.nan, "nan": np.nan, "None": np.nan, "null": np.nan, "N/A": np.nan, "na": np.nan})
        log.append("3) Cleaned spaces and normalized blank/null text")

        # 4. Detect and convert date columns
        for col in df.columns:
            if df[col].dtype == object:
                dt = pd.to_datetime(df[col], errors='coerce', dayfirst=True)
                if dt.notna().mean() > 0.75:
                    df[col] = dt
                    log.append(f"4) Converted to datetime: {col}")

        # 5. Convert numeric-looking columns
        for col in df.columns:
            if df[col].dtype == object:
                cleaned = df[col].astype(str).str.replace(',', '', regex=False).str.replace('%', '', regex=False)
                num = pd.to_numeric(cleaned, errors='coerce')
                if num.notna().mean() > 0.75:
                    df[col] = num
                    log.append(f"5) Converted to numeric: {col}")

        # 6. Fill missing values smartly
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                if df[col].isna().sum() > 0:
                    med = df[col].median()
                    df[col] = df[col].fillna(med)
                    log.append(f"6) Filled numeric missing in {col} with median")
            elif pd.api.types.is_datetime64_any_dtype(df[col]):
                if df[col].isna().sum() > 0:
                    mode = df[col].mode()
                    if not mode.empty:
                        df[col] = df[col].fillna(mode.iloc[0])
                        log.append(f"6) Filled date missing in {col} with mode")
            else:
                if df[col].isna().sum() > 0:
                    mode = df[col].mode()
                    fill_value = mode.iloc[0] if not mode.empty else "Unknown"
                    df[col] = df[col].fillna(fill_value)
                    log.append(f"6) Filled text missing in {col} with mode/Unknown")

        # 7. Outlier detection & capping for numeric columns
        num_cols = df.select_dtypes(include=np.number).columns.tolist()
        for col in num_cols:
            q1 = df[col].quantile(0.25)
            q3 = df[col].quantile(0.75)
            iqr = q3 - q1
            if iqr > 0:
                lower = q1 - 1.5 * iqr
                upper = q3 + 1.5 * iqr
                outlier_count = ((df[col] < lower) | (df[col] > upper)).sum()
                df[col] = df[col].clip(lower, upper)
                if outlier_count > 0:
                    log.append(f"7) Capped outliers in {col}: {outlier_count}")

        # 8. Drop useless columns (all same or all null)
        drop_cols = []
        for col in df.columns:
            if df[col].isna().all() or df[col].nunique(dropna=False) <= 1:
                drop_cols.append(col)
        if drop_cols:
            df.drop(columns=drop_cols, inplace=True)
            log.append(f"8) Dropped low-information columns: {', '.join(drop_cols)}")

        def finish_auto_clean():
            self.df = df
            self.filtered_df = None
            self.update_combos()
            self.show_dataframe(self.df)

            self.ai_text.delete("1.0", tk.END)
            self.ai_text.insert(tk.END, "AUTO CLEANING LOG\n")
            self.ai_text.insert(tk.END, "="*70 + "\n")
            self.ai_text.insert(tk.END, f"Original Columns:\n{old_cols}\n\n")
            self.ai_text.insert(tk.END, f"New Columns:\n{list(self.df.columns)}\n\n")
            self.ai_text.insert(tk.END, "Cleaning Actions:\n")
            for item in log:
                self.ai_text.insert(tk.END, f"- {item}\n")

            self.ai_text.insert(tk.END, "\nColumn Intelligence:\n")
            for col in self.df.columns:
                col_type = smart_column_type(self.df[col])
                is_id = detect_possible_id(self.df[col])
                self.ai_text.insert(tk.END, f"- {col}: {col_type}{' | Possible ID Column' if is_id else ''}\n")

            self.set_status("AI cleaning completed")
            messagebox.showinfo("Done", "Auto cleaning completed successfully.")

        self.run_on_ui(finish_auto_clean)

    # ===============================
    # SEARCH / FILTER / SORT
    # ===============================
    def search_data(self):
        if self.df is None:
            return
        query = self.search_var.get().strip().lower()
        if not query:
            self.show_dataframe(self.filtered_df if self.filtered_df is not None else self.df)
            return

        base_df = self.filtered_df.copy() if self.filtered_df is not None else self.df.copy()
        mask = base_df.astype(str).apply(lambda col: col.str.lower().str.contains(query, na=False, regex=False))
        result = base_df[mask.any(axis=1)]
        self.show_dataframe(result)
        self.set_status(f"Search results: {len(result)} rows")

    def apply_filter(self):
        if self.df is None:
            return
        col = self.filter_col_var.get()
        cond = self.condition_var.get()
        val = self.filter_val_var.get().strip()
        if not col or val == "":
            messagebox.showwarning("Warning", "Select a column and enter a filter value.")
            return

        df = self.df.copy()
        try:
            if cond == "contains":
                filtered = df[df[col].astype(str).str.contains(val, case=False, na=False, regex=False)]
            elif cond == "equals":
                filtered = df[df[col].astype(str).str.lower() == val.lower()]
            elif cond == "starts with":
                filtered = df[df[col].astype(str).str.lower().str.startswith(val.lower(), na=False)]
            elif cond == "ends with":
                filtered = df[df[col].astype(str).str.lower().str.endswith(val.lower(), na=False)]
            else:
                numeric_val = float(val)
                col_num = pd.to_numeric(df[col], errors='coerce')
                if cond == ">":
                    filtered = df[col_num > numeric_val]
                elif cond == "<":
                    filtered = df[col_num < numeric_val]
                elif cond == ">=":
                    filtered = df[col_num >= numeric_val]
                elif cond == "<=":
                    filtered = df[col_num <= numeric_val]
                else:
                    filtered = df

            self.filtered_df = filtered.copy()
            self.show_dataframe(self.filtered_df)
            self.set_status(f"Filter applied: {len(self.filtered_df)} rows")
        except Exception as e:
            messagebox.showerror("Filter Error", str(e))

    def clear_filter(self):
        self.filtered_df = None
        if self.df is not None:
            self.show_dataframe(self.df)
            self.set_status("Filter cleared")

    def sort_data(self, ascending=True):
        active_df = self.get_active_df()
        if active_df is None:
            return
        col = self.sort_col_var.get()
        if not col:
            return

        try:
            numeric_key = pd.to_numeric(active_df[col], errors='coerce')
            if numeric_key.notna().sum() > 0:
                active_df = active_df.assign(_sort_key=numeric_key).sort_values(by="_sort_key", ascending=ascending).drop(columns=["_sort_key"])
            else:
                active_df = active_df.sort_values(by=col, ascending=ascending)
            if self.filtered_df is not None:
                self.filtered_df = active_df
            else:
                self.df = active_df
            self.show_dataframe(active_df)
            self.set_status(f"Sorted by {col} {'ascending' if ascending else 'descending'}")
        except Exception as e:
            messagebox.showerror("Sort Error", str(e))

    # ===============================
    # ANALYSIS ENGINE
    # ===============================
    def generate_analysis(self):
        df = self.get_active_df()
        if df is None:
            self.run_on_ui(messagebox.showwarning, "Warning", "Please load data first.")
            return

        self.set_status("Generating advanced analysis...")
        report = []
        report.append("ADVANCED DATA ANALYSIS REPORT")
        report.append("="*90)
        report.append(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
        report.append("\nCOLUMN TYPES")
        report.append("-"*90)
        for col in df.columns:
            report.append(f"{col}: {smart_column_type(df[col])} | Missing: {df[col].isna().sum()} | Unique: {df[col].nunique()}")

        num_cols = df.select_dtypes(include=np.number).columns.tolist()
        cat_cols = [c for c in df.columns if c not in num_cols and not pd.api.types.is_datetime64_any_dtype(df[c])]
        date_cols = [c for c in df.columns if pd.api.types.is_datetime64_any_dtype(df[c])]

        if num_cols:
            report.append("\nNUMERICAL SUMMARY")
            report.append("-"*90)
            report.append(df[num_cols].describe().to_string())

            report.append("\nCORRELATION MATRIX")
            report.append("-"*90)
            report.append(df[num_cols].corr(numeric_only=True).round(3).to_string())

        if cat_cols:
            report.append("\nTOP CATEGORICAL INSIGHTS")
            report.append("-"*90)
            for col in cat_cols[:10]:
                report.append(f"\n{col} - Top 10 Values")
                report.append(df[col].astype(str).value_counts().head(10).to_string())

        if date_cols:
            report.append("\nDATE INSIGHTS")
            report.append("-"*90)
            for col in date_cols:
                report.append(f"{col}: Min={df[col].min()} | Max={df[col].max()}")

        # Outlier summary
        if num_cols:
            report.append("\nOUTLIER SUMMARY (IQR METHOD)")
            report.append("-"*90)
            for col in num_cols:
                q1 = df[col].quantile(0.25)
                q3 = df[col].quantile(0.75)
                iqr = q3 - q1
                if iqr > 0:
                    lower = q1 - 1.5 * iqr
                    upper = q3 + 1.5 * iqr
                    count = ((df[col] < lower) | (df[col] > upper)).sum()
                    report.append(f"{col}: {count} outliers")

        # ML-ish intelligence
        if len(num_cols) >= 2 and len(df) >= 10 and IsolationForest is not None and KMeans is not None:
            try:
                clean_num = df[num_cols].dropna().copy()
                if len(clean_num) > 10:
                    model = IsolationForest(contamination=0.05, random_state=42)
                    preds = model.fit_predict(clean_num)
                    anomaly_count = (preds == -1).sum()
                    report.append("\nANOMALY DETECTION")
                    report.append("-"*90)
                    report.append(f"Potential anomalies detected: {anomaly_count}")

                    k = min(4, max(2, len(clean_num) // 20))
                    km = KMeans(n_clusters=k, random_state=42, n_init=10)
                    clusters = km.fit_predict(clean_num)
                    report.append("\nCLUSTERING")
                    report.append("-"*90)
                    report.append(f"KMeans clusters created: {k}")
                    report.append(pd.Series(clusters).value_counts().sort_index().to_string())
            except:
                pass

        # Simple predictive relation
        if len(num_cols) >= 2 and len(df) > 20 and LinearRegression is not None and r2_score is not None:
            try:
                target = num_cols[-1]
                features = num_cols[:-1]
                train_df = df[features + [target]].dropna()
                if len(train_df) > 20:
                    X = train_df[features]
                    y = train_df[target]
                    model = LinearRegression()
                    model.fit(X, y)
                    pred = model.predict(X)
                    score = r2_score(y, pred)
                    report.append("\nPREDICTIVE TREND")
                    report.append("-"*90)
                    report.append(f"Target column used: {target}")
                    report.append(f"Approx model fit (R²): {round(score, 4)}")
                    report.append("Important feature influence:")
                    coef_series = pd.Series(model.coef_, index=features).sort_values(key=np.abs, ascending=False)
                    report.append(coef_series.to_string())
            except:
                pass

        def finish_analysis():
            self.analysis_text.delete("1.0", tk.END)
            self.analysis_text.insert(tk.END, "\n".join(report))
            self.generate_ai_story(df, num_cols, cat_cols, date_cols)
            self.set_status("Analysis completed")
            messagebox.showinfo("Done", "Advanced analysis completed.")

        self.run_on_ui(finish_analysis)

    def generate_ai_story(self, df, num_cols, cat_cols, date_cols):
        story = []
        story.append("AI BUSINESS INSIGHTS")
        story.append("="*80)
        story.append("This section explains the data in simple human language.\n")

        story.append(f"Dataset size: {df.shape[0]} rows and {df.shape[1]} columns.")
        if num_cols:
            story.append(f"Main numeric columns detected: {', '.join(num_cols[:8])}")
        if cat_cols:
            story.append(f"Main category/text columns detected: {', '.join(cat_cols[:8])}")
        if date_cols:
            story.append(f"Date columns detected: {', '.join(date_cols[:5])}")

        for col in num_cols[:5]:
            story.append(f"\n• {col}: average is {round(df[col].mean(),2)}, minimum is {round(df[col].min(),2)}, maximum is {round(df[col].max(),2)}.")

        for col in cat_cols[:3]:
            top = df[col].astype(str).value_counts().head(3)
            story.append(f"\n• {col} top categories:")
            for idx, val in top.items():
                story.append(f"   - {idx}: {val} records")

        if len(num_cols) >= 2:
            corr = df[num_cols].corr(numeric_only=True)
            corr_pairs = []
            for i in range(len(corr.columns)):
                for j in range(i+1, len(corr.columns)):
                    corr_pairs.append((corr.columns[i], corr.columns[j], corr.iloc[i, j]))
            corr_pairs = sorted(corr_pairs, key=lambda x: abs(x[2]), reverse=True)
            if corr_pairs:
                a, b, c = corr_pairs[0]
                story.append(f"\n• Strongest relationship found: {a} and {b} (correlation = {round(c, 3)}).")
                if c > 0:
                    story.append("  This means both columns tend to increase together.")
                else:
                    story.append("  This means when one increases, the other often decreases.")

        story.append("\nRECOMMENDED NEXT STEPS")
        story.append("-"*80)
        story.append("1) Focus on high-correlation features for business decisions.")
        story.append("2) Investigate outliers because they may indicate errors or important rare events.")
        story.append("3) Use filters and sorting to inspect low/high performers.")
        story.append("4) Export the cleaned dataset and use it for dashboards or ML models.")

        self.ai_text.delete("1.0", tk.END)
        self.ai_text.insert(tk.END, "\n".join(story))

    # ===============================
    # VISUALIZATION
    # ===============================
    def plot_chart(self):
        df = self.get_active_df()
        if df is None:
            return

        chart = self.chart_type.get()
        x = self.x_var.get()
        y = self.y_var.get()

        if self.colorbar is not None:
            self.colorbar.remove()
            self.colorbar = None
        self.ax.clear()
        try:
            if chart == "Histogram":
                pd.to_numeric(df[x], errors='coerce').dropna().plot(kind="hist", ax=self.ax, bins=20)
                self.ax.set_title(f"Histogram of {x}")
            elif chart == "Bar":
                df[x].astype(str).value_counts().head(10).plot(kind="bar", ax=self.ax)
                self.ax.set_title(f"Top 10 {x}")
            elif chart == "Line":
                self.ax.plot(df[x].astype(str).head(100), pd.to_numeric(df[y], errors='coerce').head(100))
                self.ax.set_title(f"Line Plot: {x} vs {y}")
                self.ax.tick_params(axis='x', rotation=45)
            elif chart == "Scatter":
                self.ax.scatter(pd.to_numeric(df[x], errors='coerce'), pd.to_numeric(df[y], errors='coerce'))
                self.ax.set_title(f"Scatter Plot: {x} vs {y}")
            elif chart == "Boxplot":
                self.ax.boxplot(pd.to_numeric(df[x], errors='coerce').dropna())
                self.ax.set_title(f"Boxplot of {x}")
            elif chart == "Pie":
                df[x].astype(str).value_counts().head(8).plot(kind="pie", ax=self.ax, autopct="%1.1f%%")
                self.ax.set_ylabel("")
                self.ax.set_title(f"Pie Chart of {x}")
            elif chart == "Correlation Heatmap":
                num = df.select_dtypes(include=np.number)
                if num.shape[1] < 2:
                    raise ValueError("Need at least 2 numeric columns for heatmap")
                corr = num.corr()
                im = self.ax.imshow(corr, aspect='auto')
                self.ax.set_xticks(range(len(corr.columns)))
                self.ax.set_yticks(range(len(corr.columns)))
                self.ax.set_xticklabels(corr.columns, rotation=45, ha='right')
                self.ax.set_yticklabels(corr.columns)
                self.ax.set_title("Correlation Heatmap")
                self.colorbar = self.fig.colorbar(im, ax=self.ax)
            elif chart == "Outliers":
                vals = pd.to_numeric(df[x], errors='coerce').dropna()
                self.ax.boxplot(vals)
                self.ax.set_title(f"Outlier Detection for {x}")

            self.fig.tight_layout()
            self.canvas.draw()
            self.current_chart = chart
            self.set_status(f"Chart plotted: {chart}")
        except Exception as e:
            messagebox.showerror("Plot Error", str(e))

    def save_chart(self):
        if self.current_chart is None:
            messagebox.showwarning("Warning", "Please plot a chart first.")
            return
        file_path = filedialog.asksaveasfilename(defaultextension=".png", filetypes=[("PNG", "*.png"), ("JPG", "*.jpg")])
        if file_path:
            self.fig.savefig(file_path, bbox_inches="tight")
            messagebox.showinfo("Saved", "Chart saved successfully.")

    # ===============================
    # SMART SET FORMAT
    # ===============================
    def open_set_format_dialog(self):
        if self.df is None:
            messagebox.showwarning("Warning", "Please load a file first.")
            return

        dialog = tk.Toplevel(self.root)
        dialog.title("Set Format")
        dialog.geometry("540x640")
        dialog.transient(self.root)
        dialog.grab_set()

        ttk.Label(
            dialog,
            text="Select Column",
            font=("Segoe UI", 11, "bold")
        ).pack(anchor="w", padx=12, pady=(12, 5))

        list_frame = ttk.Frame(dialog)
        list_frame.pack(fill="both", expand=True, padx=12)

        col_list = tk.Listbox(
            list_frame,
            height=8,
            exportselection=False
        )

        scrollbar = ttk.Scrollbar(
            list_frame,
            orient="vertical",
            command=col_list.yview
        )

        col_list.configure(yscrollcommand=scrollbar.set)

        col_list.pack(side="left", fill="both", expand=True)
        scrollbar.pack(side="right", fill="y")

        for col in self.df.columns:
            col_list.insert(tk.END, col)

        action_var = tk.StringVar(value="ai_best")
        prefix_var = tk.StringVar(value="")
        suggestion_var = tk.StringVar(
            value="Choose a column to see AI suggestion."
        )

        action_box = ttk.LabelFrame(
            dialog,
            text="Formatting Action",
            padding=10
        )
        action_box.pack(fill="x", padx=12, pady=10)

        actions = [
            ("AI Best", "ai_best"),
            ("Remove Rs/currency and make clean number", "money_to_number"),
            ("Add Rs format to all numeric values", "money_with_rs"),
            ("Remove common text prefix", "remove_prefix"),
            ("Add common text prefix", "add_prefix"),
            ("Title Case text", "title_case"),
            ("Change Date Format", "date_format"),
        ]

        for label, value in actions:
            ttk.Radiobutton(
                action_box,
                text=label,
                value=value,
                variable=action_var
            ).pack(anchor="w", pady=1)

        prefix_row = ttk.Frame(dialog)
        prefix_row.pack(fill="x", padx=12)

        
        ttk.Label(
            prefix_row,
            text="Prefix / Date Format:"
        ).pack(side="left")

        ttk.Entry(
            prefix_row,
            textvariable=prefix_var,
            width=18
        ).pack(side="left", padx=8)

        ttk.Label(
            dialog,
            textvariable=suggestion_var,
            wraplength=500,
            foreground="#1f4e79"
        ).pack(fill="x", padx=12, pady=8)

        def selected_column():
            selection = col_list.curselection()

            if not selection:
                return None

            return col_list.get(selection[0])

        def refresh_suggestion(event=None):
            col = selected_column()

            if not col:
                return

            profile = self.detect_format_profile(self.df[col])

            if profile["common_prefix"]:
                prefix_var.set(profile["common_prefix"])

            suggestion_var.set(
                profile["suggestion"] +
                "\n\nExamples:"
                "\n dd/mm/yyyy"
                "\n mm/dd/yyyy"
                "\n yyyy/mm/dd"
                "\n yyyy-mm-dd"
            )

        def apply_choice():
            col = selected_column()

            if not col:
                messagebox.showwarning(
                    "Warning",
                    "Please select a column."
                )
                return

            action = action_var.get()
            prefix = prefix_var.get().strip()

            try:
                changed = self.apply_set_format(
                    col,
                    action,
                    prefix
                )

                if changed:
                    self.show_dataframe(self.df)
                    self.update_combos()

                    dialog.destroy()

                    messagebox.showinfo(
                        "Success",
                        f"Format applied successfully on '{col}'"
                    )

            except Exception as e:
                messagebox.showerror(
                    "Format Error",
                    str(e)
                )

        col_list.bind("<<ListboxSelect>>", refresh_suggestion)

        if self.df.columns.size:
            col_list.selection_set(0)
            refresh_suggestion()

        btn_row = ttk.Frame(dialog)
        btn_row.pack(fill="x", padx=12, pady=12)

        submit_btn = ttk.Button(
            btn_row,
            text="Submit",
            command=lambda: apply_choice()
        )
        submit_btn.pack(side="right", padx=5)

        cancel_btn = ttk.Button(
            btn_row,
            text="Cancel",
            command=dialog.destroy
        )
        cancel_btn.pack(side="right")
    def detect_format_profile(self, series):
        text = series.dropna().astype(str).str.strip()
        if text.empty:
            return {"common_prefix": "", "suggestion": "This column is empty, so no format change is needed."}

        currency_hits = text.str.contains(r"(?i)(₹|rs\.?|inr|rupees?)", regex=True).mean()
        numeric_clean = text.str.replace(r"(?i)(₹|rs\.?|inr|rupees?|/-)", "", regex=True)
        numeric_clean = numeric_clean.str.replace(",", "", regex=False).str.strip()
        numeric_ratio = pd.to_numeric(numeric_clean.str.extract(r"(-?\d+(?:\.\d+)?)")[0], errors="coerce").notna().mean()

        prefixes = text.str.extract(r"^\s*([A-Za-z]+)[\s_-]*")[0].dropna().str.upper()
        common_prefix = ""
        prefix_message = ""
        if not prefixes.empty:
            common_prefix = prefixes.mode().iloc[0]
            prefix_count = int((prefixes == common_prefix).sum())
            prefix_message = f" Common prefix found: {common_prefix} in {prefix_count} values."

        title_ratio = text.str.contains(r"[A-Za-z]", regex=True).mean()
        date_test = pd.to_datetime(
            text,
            errors="coerce",
            dayfirst=True
        )

        date_ratio = date_test.notna().mean()

        if date_ratio > 0.70:
            suggestion = (
                "AI suggestion: This looks like a Date column. "
                "Use Change Date Format and enter formats like "
                "dd/mm/yyyy or yyyy-mm-dd."
            )

        
            suggestion = "AI suggestion: this looks like amount/salary data. Use AI Best to remove currency symbols, commas, and mixed int/float formatting."
        elif common_prefix:
            suggestion = f"AI suggestion: this looks like code/id text.{prefix_message} Use AI Best to make prefix usage consistent."
        elif title_ratio > 0.75:
            suggestion = "AI suggestion: this looks like names or text labels. Use AI Best to convert it into clean Title Case."
        else:
            suggestion = "AI suggestion: no strong money or prefix pattern found. Pick a manual action if you still want to force a format."
        return {"common_prefix": common_prefix, "suggestion": suggestion}

    def apply_set_format(self, col, action, prefix):
        temp_df = self.df.copy()
        old_series = temp_df[col].copy()
        new_series, log = self.smart_format_series(old_series, action, prefix)

        changed_mask = old_series.astype(str).fillna("") != new_series.astype(str).fillna("")
        changed_count = int(changed_mask.sum())
        if changed_count == 0:
            messagebox.showinfo("No Change", "No values needed formatting for this column.")
            return False

        preview_rows = []
        for idx in old_series[changed_mask].head(8).index:
            preview_rows.append(f"{old_series.loc[idx]}  ->  {new_series.loc[idx]}")
        preview = "\n".join(preview_rows)
        confirm = messagebox.askyesno("Confirm Set Format", f"Column: {col}\nChanges: {changed_count}\n\n{log}\n\nPreview:\n{preview}\n\nApply these changes?")
        if not confirm:
            return False

        temp_df[col] = new_series
        self.df = temp_df
        self.filtered_df = None
        self.update_combos()
        self.show_dataframe(self.df)
        self.ai_text.delete("1.0", tk.END)
        self.ai_text.insert(tk.END, "SET FORMAT LOG\n")
        self.ai_text.insert(tk.END, "="*70 + "\n")
        self.ai_text.insert(tk.END, f"Column: {col}\n{log}\nChanged values: {changed_count}\n")
        self.set_status(f"Set Format applied on {col}: {changed_count} values changed")
        return True

    def smart_format_series(self, series, action, prefix):
        text = series.astype("object")
        non_null = text.dropna().astype(str).str.strip()
        if non_null.empty:
            return series.copy(), "Column is empty."

        profile = self.detect_format_profile(series)
        common_prefix = prefix or profile["common_prefix"]
        money_pattern = r"(?i)(₹|rs\.?|inr|rupees?|/-)"
        numeric_clean = non_null.str.replace(money_pattern, "", regex=True).str.replace(",", "", regex=False).str.strip()
        extracted_num = pd.to_numeric(numeric_clean.str.extract(r"(-?\d+(?:\.\d+)?)")[0], errors="coerce")
        numeric_ratio = extracted_num.notna().mean()

        if action == "ai_best":
            currency_hits = non_null.str.contains(money_pattern, regex=True).mean()
            date_check = pd.to_datetime(
                non_null,
                errors="coerce",
                dayfirst=True
            )

            if date_check.notna().mean() > 0.70:
                action = "date_format"
                prefix = "dd/mm/yyyy"
            if currency_hits > 0.15 or numeric_ratio > 0.75:
                action = "money_to_number"
            elif common_prefix:
                prefix_present = non_null.str.upper().str.match(rf"^\s*{re.escape(common_prefix.upper())}[\s_-]*")
                action = "add_prefix" if prefix_present.mean() >= 0.35 and prefix_present.mean() < 1 else "remove_prefix"
            elif non_null.str.contains(r"[A-Za-z]", regex=True).mean() > 0.75:
                action = "title_case"
            else:
                return series.copy(), "AI could not find a strong repeated pattern."

        result = series.copy()
        if action == "money_to_number":
            cleaned_all = text.astype(str).str.replace(money_pattern, "", regex=True).str.replace(",", "", regex=False).str.strip()
            nums = pd.to_numeric(cleaned_all.str.extract(r"(-?\d+(?:\.\d+)?)")[0], errors="coerce")
            if nums.dropna().apply(lambda x: float(x).is_integer()).all():
                result = nums.astype("Int64")
            else:
                result = nums.round(2)
            result[series.isna()] = np.nan
            return result, "Removed Rs/currency text, commas, extra words, and converted values into clean numbers."

        if action == "money_with_rs":
            cleaned_all = text.astype(str).str.replace(money_pattern, "", regex=True).str.replace(",", "", regex=False).str.strip()
            nums = pd.to_numeric(cleaned_all.str.extract(r"(-?\d+(?:\.\d+)?)")[0], errors="coerce")
            formatted = nums.map(lambda x: f"Rs {x:,.0f}" if pd.notna(x) and float(x).is_integer() else (f"Rs {x:,.2f}" if pd.notna(x) else np.nan))
            formatted[series.isna()] = np.nan
            return formatted, "Converted all numeric-looking values into one Rs format."
        if action == "date_format":

            fmt = prefix.strip()

            format_map = {
                "dd/mm/yyyy": "%d/%m/%Y",
                "mm/dd/yyyy": "%m/%d/%Y",
                "yyyy/mm/dd": "%Y/%m/%d",

                "dd-mm-yyyy": "%d-%m-%Y",
                "mm-dd-yyyy": "%m-%d-%Y",
                "yyyy-mm-dd": "%Y-%m-%d",

                "dd/mm/yy": "%d/%m/%y",
                "mm/dd/yy": "%m/%d/%y",
                "yy/mm/dd": "%y/%m/%d"
            }

            if fmt.lower() not in format_map:
                return (
                    series.copy(),
                    "Invalid date format. Examples:\n"
                    "dd/mm/yyyy\n"
                    "mm/dd/yyyy\n"
                    "yyyy/mm/dd\n"
                    "yyyy-mm-dd"
                )

            dt = pd.to_datetime(
                series,
                errors="coerce",
                dayfirst=True
            )

            result = dt.dt.strftime(
                format_map[fmt.lower()]
            )

            result[dt.isna()] = np.nan

            return (
                result,
                f"Converted dates into format: {fmt}"
            )
        if action == "title_case":
            result = text.astype(str).str.strip().str.replace(r"\s+", " ", regex=True).str.title()
            result[series.isna()] = np.nan
            return result, "Converted text values into clean Title Case and removed extra spaces."

        if action in ["remove_prefix", "add_prefix"]:
            if not common_prefix:
                return series.copy(), "No common prefix found. Enter a prefix manually."
            prefix_regex = rf"(?i)^\s*{re.escape(common_prefix)}[\s_-]*"
            values = text.astype(str).str.strip()
            if action == "remove_prefix":
                result = values.str.replace(prefix_regex, "", regex=True).str.strip()
                result[series.isna()] = np.nan
                return result, f"Removed common prefix '{common_prefix}' from this column."
            base = values.str.replace(prefix_regex, "", regex=True).str.strip()
            result = base.map(lambda x: f"{common_prefix}{x}" if x and x.lower() != "nan" else np.nan)
            result[series.isna()] = np.nan
            return result, f"Added common prefix '{common_prefix}' to values that were missing it."

        return series.copy(), "Unknown format action."

    # ===============================
    # EXPORT / RESET
    # ===============================
    def export_clean_data(self):
        if self.df is None:
            return
        file_path = filedialog.asksaveasfilename(defaultextension=".csv", filetypes=[("CSV", "*.csv"), ("Excel", "*.xlsx")])
        if not file_path:
            return
        try:
            export_df = self.get_active_df()
            if file_path.endswith(".csv"):
                export_df.to_csv(file_path, index=False)
            else:
                export_df.to_excel(file_path, index=False)
            messagebox.showinfo("Exported", "Cleaned data exported successfully.")
            self.set_status("Export completed")
        except Exception as e:
            messagebox.showerror("Export Error", str(e))

    def reset_data(self):
        if self.original_df is not None:
            self.df = self.original_df.copy()
            self.filtered_df = None
            self.update_combos()
            self.show_dataframe(self.df)
            self.analysis_text.delete("1.0", tk.END)
            self.ai_text.delete("1.0", tk.END)
            self.set_status("Data reset to original")


if __name__ == "__main__":
    root = tk.Tk()
    app = AIDataStudio(root)
    root.mainloop()